# Forecasting de precios de berries — TDA vs. sin TDA (versión corregida)

Notebook completo y autocontenido. Corrige tres problemas del enfoque original:

1. **El target ya no está dentro de la ventana.** Antes `SlidingWindow` devolvía como
   objetivo el *último valor de la ventana* (el modelo "predecía" un dato que ya tenía
   como entrada → era *nowcasting*/persistencia, no forecasting). Aquí el objetivo es
   `signal[t]`, **fuera** de la ventana `signal[t-w:t]`.
2. **Se entrena con toda la historia** y se evalúa en el bloque contiguo más reciente,
   en lugar de entrenar en un régimen (p. ej. 2008-2012) y testear en otro (2020-2022).
3. **Se modela la diferencia** (cambio mes a mes), que es ~estacionaria; modelar el nivel
   con tendencia empuja a todos los modelos a la persistencia y desestabiliza al MLP.

Además se compara **siempre contra el baseline ingenuo** (mañana = hoy): un modelo solo
"sirve" si su MAE queda por debajo del ingenuo.

> Ajusta `CSV_PATH` en la primera celda. El resto corre de arriba a abajo.

In [ ]:
# ============================================================
# 1. CONFIGURACIÓN E IMPORTS
# ============================================================
CSV_PATH = "WPUSI01102B.csv"   # <-- AJUSTA esta ruta a tu archivo
N_TEST   = 18                  # meses recientes reservados para prueba
WINDOW   = 24                  # tamaño de ventana (meses)
DIM, TAU = 3, 1                # parámetros TDA (embedding_dim, time_delay)
DIFF     = True                # modelar la diferencia (recomendado)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence

from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

plt.rcParams['figure.figsize'] = (12, 6); plt.rcParams['font.size'] = 11
print("✅ Imports listos")

In [ ]:
# ============================================================
# 2. CARGA DE DATOS
# ============================================================
df = pd.read_csv(CSV_PATH, parse_dates=['observation_date'])
precios = df['WPUSI01102B'].values.astype(float)
fechas  = df['observation_date'].values

print(f"Registros: {len(precios)} | {str(fechas[0])[:7]} a {str(fechas[-1])[:7]}")
print(f"Media={precios.mean():.1f}  Std={precios.std():.1f}  "
      f"Min={precios.min():.1f}  Max={precios.max():.1f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(fechas, precios, 'b-', lw=1.5, alpha=0.8)
ax.axhline(100, color='red', ls='--', alpha=0.5, label='Base 1982=100')
ax.set_title('PPI Berries (WPUSI01102B)'); ax.set_ylabel('Índice PPI')
ax.grid(True, alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 3. FUNCIONES DE FORECASTING (sin fuga de información)
# ============================================================
def _tda_feats(win, dim=DIM, tau=TAU):
    '''4 características topológicas (H1) de una ventana 1D.'''
    try:
        emb = SingleTakensEmbedding(parameters_type="fixed", dimension=dim,
                                    time_delay=tau).fit_transform(win)
        if len(emb) >= 5:
            d = VietorisRipsPersistence(homology_dimensions=[1], n_jobs=-1,
                                        collapse_edges=True).fit_transform([emb])[0]
            m = d[:, 2] == 1
            if np.any(m):
                p = d[m, 1] - d[m, 0]
                return [np.max(p), np.mean(p), np.std(p), len(p)]
        return [0, 0, 0, 0]
    except Exception:
        return [0, 0, 0, 0]

def construir_Xy(signal, window_size, use_tda=False, embedding_dim=DIM, time_delay=TAU):
    '''Ventana = signal[t-w:t]; target = signal[t] (el SIGUIENTE valor, fuera de la ventana).'''
    Xr, y, topo = [], [], []
    for t in range(window_size, len(signal)):
        win = signal[t-window_size:t]
        Xr.append(win); y.append(signal[t])
        if use_tda:
            topo.append(_tda_feats(win, embedding_dim, time_delay))
    return np.array(Xr), np.array(y), (np.array(topo) if use_tda else None)

def backtest(signal, window_size=WINDOW, n_test=N_TEST, use_tda=False, modelo=None,
             embedding_dim=DIM, time_delay=TAU, diferenciar=DIFF, retornar_preds=False):
    '''Entrena con toda la historia menos los ultimos n_test meses y evalua en ellos.
       diferenciar=True: modela el cambio mes a mes y reconstruye el nivel.'''
    if modelo is None:
        modelo = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

    serie = np.diff(signal) if diferenciar else signal
    Xr, y, topo = construir_Xy(serie, window_size, use_tda, embedding_dim, time_delay)
    n = len(y); split = n - n_test

    sc = StandardScaler().fit(Xr[:split]); Xr_n = sc.transform(Xr)
    if use_tda:
        sct = StandardScaler().fit(topo[:split]); X = np.hstack([Xr_n, sct.transform(topo)])
    else:
        X = Xr_n

    m = clone(modelo); m.fit(X[:split], y[:split])
    pred = m.predict(X[split:]); real = y[split:]

    if diferenciar:                                   # reconstruir nivel
        base_idx = window_size + split
        prev = signal[base_idx : base_idx + n_test]
        pred = prev + pred
        real = signal[base_idx + 1 : base_idx + 1 + n_test]
        L = min(len(pred), len(real)); pred, real = pred[:L], real[:L]
        naive = prev[:L]
    else:
        naive = Xr[split:, -1]

    out = {'mae': mean_absolute_error(real, pred), 'r2': r2_score(real, pred),
           'mae_naive': mean_absolute_error(real, naive),
           'r2_naive': r2_score(real, naive)}
    if retornar_preds:
        out.update({'real': real, 'pred': pred, 'naive': naive})
    return out

print("✅ Funciones listas (target futuro + diferenciación + baseline ingenuo)")

In [ ]:
# ============================================================
# 4. CATÁLOGO DE MODELOS
# ============================================================
modelos = {
    "Random Forest":  RandomForestRegressor(n_estimators=200, max_depth=10,
                                             min_samples_split=5, random_state=42, n_jobs=-1),
    "Extra Trees":    ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Grad. Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                                learning_rate=0.05, random_state=42),
    "Hist GB":        HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05,
                                                    random_state=42),
    "Ridge":          Ridge(alpha=1.0),
    "Lasso":          Lasso(alpha=0.1, max_iter=10000),
    "ElasticNet":     ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
    "SVR (RBF)":      SVR(kernel="rbf", C=10, gamma="scale"),
    "KNN":            KNeighborsRegressor(n_neighbors=5, weights="distance"),
    "MLP":            MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000,
                                   random_state=42, early_stopping=True),
}
try:
    from xgboost import XGBRegressor
    modelos["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                      random_state=42, n_jobs=-1, verbosity=0)
except Exception:
    print("ℹ️ XGBoost no instalado (pip install xgboost)")
try:
    from lightgbm import LGBMRegressor
    modelos["LightGBM"] = LGBMRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                        random_state=42, n_jobs=-1, verbose=-1)
except Exception:
    print("ℹ️ LightGBM no instalado (pip install lightgbm)")

print(f"✅ {len(modelos)} modelos:", ", ".join(modelos))

In [ ]:
# ============================================================
# 5. COMPARATIVA PRINCIPAL: cada modelo, con/sin TDA, vs ingenuo
# ============================================================
filas = []; ref_naive = None
for nombre, modelo in modelos.items():
    sin = backtest(precios, use_tda=False, modelo=modelo)
    con = backtest(precios, use_tda=True,  modelo=modelo)
    ref_naive = sin['mae_naive']
    filas.append({'Modelo': nombre,
                  'MAE_sinTDA': sin['mae'], 'R2_sinTDA': sin['r2'],
                  'MAE_conTDA': con['mae'], 'R2_conTDA': con['r2'],
                  'mejora_%': (sin['mae'] - con['mae']) / sin['mae'] * 100,
                  'gana_al_naive': con['mae'] < ref_naive})

df_res = pd.DataFrame(filas).sort_values('MAE_conTDA').reset_index(drop=True)
print(f"BASELINE INGENUO (mañana=hoy):  MAE = {ref_naive:.3f}\n")
print(df_res.round(3).to_string(index=False))

# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
orden = df_res['Modelo'].tolist(); x = np.arange(len(orden)); w = 0.38
ax = axes[0]
ax.bar(x - w/2, df_res['MAE_sinTDA'], w, label='Sin TDA', color='coral', ec='black', alpha=0.85)
ax.bar(x + w/2, df_res['MAE_conTDA'], w, label='Con TDA', color='steelblue', ec='black', alpha=0.85)
ax.axhline(ref_naive, color='black', ls='--', lw=1.5, label=f'Ingenuo ({ref_naive:.1f})')
ax.set_xticks(x); ax.set_xticklabels(orden, rotation=45, ha='right')
ax.set_ylabel('MAE'); ax.set_title('MAE por modelo (debajo de la línea = mejor que el ingenuo)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
mej = df_res['mejora_%']; colors = ['green' if m > 0 else 'red' for m in mej]
ax.bar(x, mej, color=colors, ec='black', alpha=0.85)
ax.axhline(0, color='black', lw=1.2)
ax.set_xticks(x); ax.set_xticklabels(orden, rotation=45, ha='right')
ax.set_ylabel('Mejora del TDA en MAE (%)'); ax.set_title('¿Cuánto aporta el TDA en cada modelo?')
ax.grid(True, alpha=0.3, axis='y')
for xi, m in zip(x, mej):
    ax.text(xi, m + (0.2 if m > 0 else -0.5), f'{m:+.1f}%', ha='center', fontsize=8, fontweight='bold')
plt.suptitle(f'Comparativa — Entreno hasta {N_TEST} meses antes del final, prueba en los últimos {N_TEST}',
             fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 6. ESTABILIDAD ENTRE DISTINTOS CORTES DE PRUEBA (walk-forward)
# ============================================================
# Repite el backtest moviendo el final del periodo de prueba hacia atrás.
# Un modelo robusto mantiene MAE bajo en todos los cortes.
cortes = [0, 12, 24, 36]          # meses a recortar del final antes de tomar el test
modelos_clave = ["Random Forest", "Ridge", "SVR (RBF)", "MLP"]

print(f"MAE (con TDA) por corte | test = {N_TEST} meses terminando 'k' meses antes del final\n")
print(f"{'Modelo':<16}" + "".join([f"corte-{c:<6}" for c in cortes]))
print("-" * 60)
estab = {}
for nombre in modelos_clave:
    fila = []
    for c in cortes:
        s = precios[:len(precios) - c] if c > 0 else precios
        r = backtest(s, use_tda=True, modelo=modelos[nombre])
        fila.append(r['mae'])
    estab[nombre] = fila
    print(f"{nombre:<16}" + "".join([f"{v:<12.2f}" for v in fila]))

fig, ax = plt.subplots(figsize=(10, 5))
for nombre, fila in estab.items():
    ax.plot(cortes, fila, 'o-', lw=2, label=nombre)
ax.set_xlabel('Meses recortados del final (corte)'); ax.set_ylabel('MAE (con TDA)')
ax.set_title('Estabilidad del MAE entre distintos cortes de prueba')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 7. ESTABILIDAD DEL MLP: varias semillas (con vs sin diferenciar)
# ============================================================
SEEDS = list(range(10))
def mlp(s): return MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000,
                                random_state=s, early_stopping=True)

res = {'nivel (DIFF=False)': [], 'diferencia (DIFF=True)': []}
for s in SEEDS:
    res['nivel (DIFF=False)'].append(
        backtest(precios, use_tda=True, modelo=mlp(s), diferenciar=False)['mae'])
    res['diferencia (DIFF=True)'].append(
        backtest(precios, use_tda=True, modelo=mlp(s), diferenciar=True)['mae'])

for k, v in res.items():
    v = np.array(v)
    print(f"MLP {k:<24} MAE: media={v.mean():7.2f}  std={v.std():6.2f}  "
          f"rango=[{v.min():.1f}, {v.max():.1f}]")

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(list(res.values()), labels=list(res.keys()), patch_artist=True)
for b, c in zip(bp['boxes'], ['salmon', 'steelblue']): b.set_facecolor(c)
ax.set_ylabel('MAE del MLP'); ax.set_title('Diferenciar estabiliza al MLP (menor dispersión entre semillas)')
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 8. PRONÓSTICO DEL MEJOR MODELO: real vs predicho vs ingenuo
# ============================================================
mejor = df_res.iloc[0]['Modelo']
print(f"Mejor modelo por MAE con TDA: {mejor}")

r = backtest(precios, use_tda=True, modelo=modelos[mejor], retornar_preds=True)
rb = backtest(precios, use_tda=False, modelo=modelos[mejor], retornar_preds=True)
L = len(r['real']); ejex = fechas[-L:]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ejex, r['real'],  'b-o', lw=2.5, ms=7, label='Real')
ax.plot(ejex, r['pred'],  'g--^', lw=2, ms=6, label=f'{mejor} con TDA (MAE={r["mae"]:.2f})')
ax.plot(ejex, rb['pred'], 'm--v', lw=2, ms=6, label=f'{mejor} sin TDA (MAE={rb["mae"]:.2f})')
ax.plot(ejex, r['naive'], color='gray', ls=':', lw=2, label=f'Ingenuo (MAE={r["mae_naive"]:.2f})')
ax.set_ylabel('Índice PPI'); ax.set_title(f'Pronóstico en los últimos {L} meses — {mejor}')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Cómo leer esto en tu reporte

- **R² positivo** en la tabla principal indica que ahora hay pronóstico real (con el
  montaje anterior salían negativos porque el modelo solo copiaba el último valor).
- **`gana_al_naive`**: la prueba de fuego. Si un modelo no baja del MAE ingenuo, no aporta,
  tenga o no TDA.
- **`mejora_%`**: efecto del TDA, ahora medido sobre un problema no trivial. Suele ser
  pequeño y variar por modelo; preséntalo junto al MAE absoluto, nunca solo.
- **Estabilidad entre cortes (celda 6)** y **del MLP entre semillas (celda 7)**: respaldan
  por qué confías (o no) en cada modelo. El MLP sobre *diferencias* deja de ser inestable.

Extensión opcional: añadir un SARIMA/ETS de `statsmodels` como referencia clásica de
series de tiempo, y/o `GridSearchCV` con `TimeSeriesSplit` para afinar hiperparámetros
respetando el orden temporal.